## Simple PyTorch Linear Model for Knockdown Prediction
This section trains a basic linear model that uses a non-targeting expression profile plus a target gene label to predict the knocked-down expression spectrum.

### Requirements
This notebook uses `torch` for the linear regression model. If PyTorch is not yet installed in the project virtual environment, install it with:

```bash
source .venv/bin/activate
pip install torch
```


In [8]:
import scanpy as sc
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from sklearn.model_selection import StratifiedKFold
from scipy.stats import pearsonr
from pathlib import Path

# Load the processed data
data_path = Path('processed_adata.h5ad')
if not data_path.exists():
    data_path = Path('baseline') / 'processed_adata.h5ad'
if not data_path.exists():
    data_path = Path('zyt') / 'processed_adata.h5ad'
if not data_path.exists():
    raise FileNotFoundError('Could not find processed_adata.h5ad in current, baseline, or zyt folder')

print('Loading data from:', data_path)
adata = sc.read_h5ad(data_path)
print('adata shape:', adata.shape)

# Ensure dense float32 data for PyTorch
X = adata.X
if hasattr(X, 'toarray'):
    X = X.toarray()
X = X.astype(np.float32)

# Calculate global mean control template
control_mask = adata.obs['target_gene'] == 'non-targeting'
control_expr = X[control_mask.values]
control_template = control_expr.mean(axis=0).astype(np.float32)

# Verify target genes are in var_names
target_genes_all = [g for g in adata.obs['target_gene'].cat.categories if g != 'non-targeting']
missing_genes = [g for g in target_genes_all if g not in adata.var_names]
if missing_genes:
    print(f'Warning: {len(missing_genes)} target genes missing from output features: {missing_genes}')
else:
    print('All target genes are present in the output features.')

# Prepare inputs and targets
perturb_mask = adata.obs['target_gene'] != 'non-targeting'
perturb_indices = np.where(perturb_mask.values)[0]
gene_to_idx = {gene: idx for idx, gene in enumerate(target_genes_all)}

inputs = []
targets = []
# Using a fixed seed for reproducibility in input generation
rng = np.random.default_rng(42)

for i in perturb_indices:
    gene = adata.obs['target_gene'].iat[i]
    one_hot = np.zeros(len(target_genes_all), dtype=np.float32)
    one_hot[gene_to_idx[gene]] = 1.0
    
    # Input is control_template + one_hot encoding of the perturbation
    inputs.append(np.concatenate([control_template, one_hot]))
    targets.append(X[i])

inputs = np.stack(inputs)
targets = np.stack(targets)
print('inputs shape:', inputs.shape)
print('targets shape:', targets.shape)

class KnockdownDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

all_dataset = KnockdownDataset(inputs, targets)
labels = adata.obs['target_gene'].cat.codes[perturb_indices].values

def calculate_metrics(y_true, y_pred):
    mse = np.mean((y_true - y_pred)**2)
    corrs = []
    for i in range(len(y_true)):
        if np.std(y_true[i]) > 0 and np.std(y_pred[i]) > 0:
            c, _ = pearsonr(y_true[i], y_pred[i])
            corrs.append(c)
        else:
            corrs.append(0.0)
    return mse, np.mean(corrs)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

input_dim = inputs.shape[1]
output_dim = targets.shape[1]
num_epochs = 20
batch_size = 64

fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(inputs, labels), start=1):
    print(f'\n=== Fold {fold}/{n_splits} ===')
    train_loader = DataLoader(Subset(all_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(all_dataset, val_idx), batch_size=batch_size, shuffle=False)

    model = nn.Linear(input_dim, output_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x_batch.size(0)
        train_loss /= len(train_loader.dataset)

        # Quick validation loss for tracking progress
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                loss = criterion(model(x_batch), y_batch)
                val_loss += loss.item() * x_batch.size(0)
        val_loss /= len(val_loader.dataset)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch}/{num_epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}')

    # Evaluation
    model.eval()
    val_preds = []
    val_targets = []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            val_preds.append(model(x_batch.to(device)).cpu().numpy())
            val_targets.append(y_batch.numpy())
    
    val_preds = np.concatenate(val_preds)
    val_targets = np.concatenate(val_targets)
    
    mse, corr = calculate_metrics(val_targets, val_preds)
    
    # Baseline: Predict Mean Control for everyone
    mean_ctrl_preds = np.tile(control_template, (len(val_targets), 1))
    base_mse, base_corr = calculate_metrics(val_targets, mean_ctrl_preds)
    
    fold_metrics.append({
        'mse': mse, 'corr': corr,
        'base_mse': base_mse, 'base_corr': base_corr
    })
    print(f'Fold {fold} | Model: MSE={mse:.4f}, Corr={corr:.4f} | Base: MSE={base_mse:.4f}, Corr={base_corr:.4f}')

# Final Results
avg_mse = np.mean([f['mse'] for f in fold_metrics])
avg_corr = np.mean([f['corr'] for f in fold_metrics])
avg_base_mse = np.mean([f['base_mse'] for f in fold_metrics])
avg_base_corr = np.mean([f['base_corr'] for f in fold_metrics])

print(f'\nFinal Results (Average over {n_splits} folds):')
print(f'Linear Model   - MSE: {avg_mse:.4f}, Pearson Corr: {avg_corr:.4f}')
print(f'Mean Control   - MSE: {avg_base_mse:.4f}, Pearson Corr: {avg_base_corr:.4f}')
print(f'Improvement    - MSE: {avg_base_mse - avg_mse:.4f}, Corr: {avg_corr - avg_base_corr:.4f}')

# Save weights to the same directory as the data
save_path = data_path.parent / "knockdown_linear_model.pth"
torch.save(model.state_dict(), save_path)
print(f'\nModel weights saved to {save_path}')

Loading data from: processed_adata.h5ad
adata shape: (19771, 3040)
All target genes are present in the output features.
inputs shape: (7758, 3090)
targets shape: (7758, 3040)
Using device: cpu

=== Fold 1/5 ===


/var/folders/t1/md26m3vs5mldtk18qm327l6h0000gp/T/ipykernel_22434/2335538636.py:76: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  labels = adata.obs['target_gene'].cat.codes[perturb_indices].values


  Epoch 1/20 | Train Loss: 1.052383 | Val Loss: 1.062907
  Epoch 5/20 | Train Loss: 1.044441 | Val Loss: 1.060603
  Epoch 10/20 | Train Loss: 1.040083 | Val Loss: 1.060516
  Epoch 15/20 | Train Loss: 1.038250 | Val Loss: 1.060730
  Epoch 20/20 | Train Loss: 1.037285 | Val Loss: 1.061428
Fold 1 | Model: MSE=1.0614, Corr=0.0596 | Base: MSE=1.0648, Corr=-0.0282

=== Fold 2/5 ===
  Epoch 1/20 | Train Loss: 1.056644 | Val Loss: 1.046035
  Epoch 5/20 | Train Loss: 1.048577 | Val Loss: 1.043788
  Epoch 10/20 | Train Loss: 1.044329 | Val Loss: 1.043272
  Epoch 15/20 | Train Loss: 1.042428 | Val Loss: 1.043750
  Epoch 20/20 | Train Loss: 1.041555 | Val Loss: 1.044612
Fold 2 | Model: MSE=1.0446, Corr=0.0585 | Base: MSE=1.0475, Corr=-0.0257

=== Fold 3/5 ===
  Epoch 1/20 | Train Loss: 1.054882 | Val Loss: 1.052733
  Epoch 5/20 | Train Loss: 1.046923 | Val Loss: 1.050462
  Epoch 10/20 | Train Loss: 1.042589 | Val Loss: 1.049927
  Epoch 15/20 | Train Loss: 1.040640 | Val Loss: 1.050740
  Epoch 20/2